In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup, AutoConfig
from tqdm import tqdm

import optuna

import warnings
warnings.filterwarnings('ignore')

/Users/oleksandrnovokhatskyi/Documents/My_new_work/Winstars.AI/NER_CV/.env_cv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## EDA

In [2]:
ner_data_path = Path('../data/ner/animals_ner_dataset.csv')
df = pd.read_csv(ner_data_path)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2582 entries, 0 to 2581
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sentence  2582 non-null   str  
 1   animal    2582 non-null   str  
dtypes: str(2)
memory usage: 40.5 KB


In [3]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.info()

<class 'pandas.DataFrame'>
Index: 2503 entries, 0 to 2526
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sentence  2503 non-null   str  
 1   animal    2503 non-null   str  
dtypes: str(2)
memory usage: 58.7 KB


In [4]:
df.head()

,sentence,animal
0,This scene includes a butterfly resting on a y...,butterfly
1,The shot focuses on a butterfly resting on a y...,butterfly
2,You can see a butterfly resting on a yellow fl...,butterfly
3,The camera catches a butterfly resting on a ye...,butterfly
4,The subject of the photo is a butterfly restin...,butterfly


In [5]:
df['animal'].value_counts()

animal
butterfly    251
cat          251
cow          251
dog          251
horse        251
sheep        250
spider       250
chicken      249
elephant     249
squirrel     249
animal         1
Name: count, dtype: int64

In [6]:
df[df['animal'] == 'animal']

,sentence,animal
1940,sentence,animal


In [7]:
df = df[df['animal'] != 'animal']
df['animal'].value_counts()

animal
butterfly    251
cat          251
cow          251
dog          251
horse        251
sheep        250
spider       250
chicken      249
elephant     249
squirrel     249
Name: count, dtype: int64

In [8]:
# Split the dataset into train, validation, and test sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['animal'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['animal'])
print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

Train set size: 2001
Validation set size: 250
Test set size: 251


## NER label preparation

In [9]:
# Function to create NER examples
def make_ner_example(sentence, animal):
    tokens = re.findall(r'\w+|[^\w\s]', sentence)
    labels = ['O'] * len(tokens)

    animal_lower = animal.lower()
    for idx, token in enumerate(tokens):
        if token.lower() == animal_lower:
            labels[idx] = 'B-ANIMAL'

    return tokens, labels


In [10]:
tokens, labels = make_ner_example("The cat is on the roof.", "cat")
print("Tokens:", tokens)
print("Labels:", labels)

Tokens: ['The', 'cat', 'is', 'on', 'the', 'roof', '.']
Labels: ['O', 'B-ANIMAL', 'O', 'O', 'O', 'O', 'O']


In [11]:
train_df[['tokens', 'labels']] = train_df.apply(
    lambda row: make_ner_example(row['sentence'], 
    row['animal']), axis=1, result_type='expand'
)
val_df[['tokens', 'labels']] = val_df.apply(
    lambda row: make_ner_example(row['sentence'], 
    row['animal']), axis=1, result_type='expand'
)
test_df[['tokens', 'labels']] = test_df.apply(
    lambda row: make_ner_example(row['sentence'], 
    row['animal']), axis=1, result_type='expand'
)

print(train_df.head())

                                               sentence   animal  \
1245  In the photo, a horse is standing beside a sto...    horse   
390   The subject of the photo is a chicken standing...  chicken   
458   The image captures the chicken foraging near a...  chicken   
511   The picture features a chicken resting beside ...  chicken   
1551  You can see a spider waiting near a wall crack...   spider   

                                                 tokens  \
1245  [In, the, photo, ,, a, horse, is, standing, be...   
390   [The, subject, of, the, photo, is, a, chicken,...   
458   [The, image, captures, the, chicken, foraging,...   
511   [The, picture, features, a, chicken, resting, ...   
1551  [You, can, see, a, spider, waiting, near, a, w...   

                                                 labels  
1245  [O, O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O,...  
390   [O, O, O, O, O, O, O, B-ANIMAL, O, O, O, O, O,...  
458   [O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O, O,...  
511 

## Label encoding and model initialization

In [12]:
# Create label mappings
label_to_id = {'O': 0, 'B-ANIMAL': 1}
id_to_label = {0: 'O', 1: 'B-ANIMAL'}

In [13]:
# 
train_df['label_ids'] = train_df['labels'].apply(lambda labels: [label_to_id[label] for label in labels])
val_df['label_ids'] = val_df['labels'].apply(lambda labels: [label_to_id[label] for label in labels])
test_df['label_ids'] = test_df['labels'].apply(lambda labels: [label_to_id[label] for label in labels])

In [14]:
train_df.head()

,sentence,animal,tokens,labels,label_ids
1245,"In the photo, a horse is standing beside a sto...",horse,"[In, the, photo, ,, a, horse, is, standing, be...","[O, O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O,...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
390,The subject of the photo is a chicken standing...,chicken,"[The, subject, of, the, photo, is, a, chicken,...","[O, O, O, O, O, O, O, B-ANIMAL, O, O, O, O, O,...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
458,The image captures the chicken foraging near a...,chicken,"[The, image, captures, the, chicken, foraging,...","[O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O, O,...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
511,The picture features a chicken resting beside ...,chicken,"[The, picture, features, a, chicken, resting, ...","[O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O, O,...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1551,You can see a spider waiting near a wall crack...,spider,"[You, can, see, a, spider, waiting, near, a, w...","[O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O, O,...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [15]:
model_name = 'bert-base-cased'
num_classes = len(label_to_id)

config = AutoConfig.from_pretrained(model_name, num_labels=num_classes)
tokenizer = AutoTokenizer.from_pretrained(model_name)

## Tokenization and label alignment

In [16]:
max_length = 32

def tokenize_and_align_labels(tokens, label_ids):
    tokenized_inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    token_to_word = tokenized_inputs.word_ids()

    aligned_labels = []
    previous_word_id = None

    for word_id in token_to_word:
        if word_id is None:
            aligned_labels.append(-100)
        elif word_id != previous_word_id:
            aligned_labels.append(label_ids[word_id])
        else:
            aligned_labels.append(-100)

        previous_word_id = word_id

    return {
        "input_ids": tokenized_inputs["input_ids"],
        "attention_mask": tokenized_inputs["attention_mask"],
        "labels": aligned_labels
    }

In [17]:
# Tokenize and align labels for the datasets
train_encoded = train_df.apply(
    lambda row: tokenize_and_align_labels(row['tokens'], 
    row['label_ids']), 
    axis=1
)

val_encoded = val_df.apply(
    lambda row: tokenize_and_align_labels(row['tokens'], 
    row['label_ids']), 
    axis=1
)

test_encoded = test_df.apply(
    lambda row: tokenize_and_align_labels(row['tokens'], 
    row['label_ids']), 
    axis=1
)

In [18]:
train_encoded[0]

{'input_ids': [101,
  1188,
  2741,
  2075,
  170,
  11057,
  8137,
  1113,
  170,
  3431,
  7366,
  1107,
  2991,
  13258,
  119,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'labels': [-100,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100,
  -100]}

## NER Dataset and DataLoader

In [19]:
class NERDataset(Dataset):
    def __init__(self, encoded_data):
        self.encoded_data = encoded_data.reset_index(drop=True)

    def __len__(self):
        return len(self.encoded_data)

    def __getitem__(self, idx):
        item = self.encoded_data.iloc[idx]

        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long)
        }

In [20]:
# Create PyTorch datasets
train_dataset = NERDataset(train_encoded)
val_dataset = NERDataset(val_encoded)
test_dataset = NERDataset(test_encoded)

train_dataset[0]

{'input_ids': tensor([ 101, 1130, 1103, 6307,  117,  170, 3241, 1110, 2288, 3148,  170, 2576,
         2095, 1223, 3999, 2379, 1609,  119,  102,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]),
 'labels': tensor([-100,    0,    0,    0,    0,    0,    1,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100])}

In [21]:
batch_size = 16
num_workers = 0
num_epochs = 3
learning_rate = 5e-5
weight_decay = 0.01

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=num_workers
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers
)

In [22]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 32])
torch.Size([16, 32])
torch.Size([16, 32])


## Model initialization

In [23]:
def build_model(model_name, num_classes):
    config = AutoConfig.from_pretrained(model_name, num_labels=num_classes)
    model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)
    return model

In [24]:
def create_optimizer(model, learning_rate, weight_decay):
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    return optimizer

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_model(model_name, num_classes).to(device)
optimizer = create_optimizer(model, learning_rate=2e-5, weight_decay=0.01)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7902.43it/s]
BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expec

### Train

In [26]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(dataloader)
    return avg_loss

In [27]:
@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        preds = preds.cpu().numpy()
        labels = labels.cpu().numpy()

        for pred_seq, label_seq in zip(preds, labels):
            for pred, label in zip(pred_seq, label_seq):
                if label != -100:
                    all_preds.append(pred)
                    all_labels.append(label)

    return all_preds, all_labels

In [38]:
best_val_f1 = 0.0
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}")

    val_preds, val_labels = evaluate(model, val_loader, device)
    val_accuracy = accuracy_score(val_labels, val_preds)
    val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(val_labels, val_preds, average='weighted')
    print(f"Validation Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1 Score: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        save_path = "../models/ner_bert/best_ner_model.pth"
        torch.save(model.state_dict(), save_path)
        print("Best model saved!")

Training: 100%|██████████| 126/126 [01:12<00:00,  1.73it/s]


Epoch 1/3, Train Loss: 0.0002


Evaluating: 100%|██████████| 16/16 [00:02<00:00,  7.22it/s]


Validation Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000
Best model saved!


Training: 100%|██████████| 126/126 [01:18<00:00,  1.60it/s]


...

Validation Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000


Training: 100%|██████████| 126/126 [01:19<00:00,  1.59it/s]


Epoch 3/3, Train Loss: 0.0000


Evaluating: 100%|██████████| 16/16 [00:02<00:00,  7.16it/s]

Validation Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000


### Test

In [39]:
model_load_path = "../models/ner_bert/best_ner_model.pth"
model.load_state_dict(torch.load(model_load_path))
model.to(device)

test_preds, test_labels = evaluate(model, test_loader, device)
test_accuracy = accuracy_score(test_labels, test_preds)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='weighted')

print(
    f"Test Accuracy: {test_accuracy:.4f}, "
    f"Precision: {test_precision:.4f}, "
    f"Recall: {test_recall:.4f}, "
    f"F1 Score: {test_f1:.4f}"
) 

Evaluating: 100%|██████████| 16/16 [00:02<00:00,  7.64it/s]

Test Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1 Score: 1.0000


In [41]:
with open("../models/ner_bert/label_to_id.json", "w") as f:
    json.dump(label_to_id, f, indent=4)

id_to_label = {str(v): k for k, v in label_to_id.items()}
with open("../models/ner_bert/id_to_label.json", "w") as f:
    json.dump(id_to_label, f, indent=4)

tokenizer.save_pretrained("../models/ner_bert/tokenizer")

('../models/ner_bert/tokenizer/tokenizer_config.json',
 '../models/ner_bert/tokenizer/tokenizer.json')